# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task.

## Packages and Data

In [1]:
# packages

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset
from src.fine_tuning.modeling import FineTuneLLM

In [2]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS
from src.config import GENERATION_MAX_LENGTH, GENERATION_NUM_BEAMS

In [3]:
# get data
sf = ScryfallDataset(task = TASK)

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 2042
	Validation Records = 511
	Test Records = 50
	Records saved to...
		../data/scryfall_seq2seq_train.json
		../data/scryfall_seq2seq_val.json
		../data/scryfall_seq2seq_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Seq2Seq Dataset Loaded
	Train Records = 2042
	Val Records = 511
	Test Records = 50
	Count Unique Tags = 0


## Modeling

In [4]:
# from transformers import Trainer
# from torch.nn import BCEWithLogitsLoss

# class CustomTrainer(Trainer):
#     def __init__(self, pos_weights, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.pos_weights = pos_weights

#     def compute_loss(self, model, inputs, return_outputs = False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs) 
#         logits = outputs.logits

#         loss_fct = BCEWithLogitsLoss(pos_weight = self.pos_weights)
#         loss = loss_fct(logits, labels)

#         return (loss, outputs) if return_outputs else loss

In [5]:
# fine tune the model
tagger = FineTuneLLM(
    model_name = MODEL_NAME,
    dataset = sf.dataset
)
tagger.prepare_data(
    max_input_length = MAX_INPUT_LENGTH,
    max_target_length = MAX_TARGET_LENGTH
)
tagger.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    generation_max_length = GENERATION_MAX_LENGTH,
    generation_num_beams = GENERATION_NUM_BEAMS
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/2042 [00:00<?, ? examples/s]

Map:   0%|          | 0/511 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


In [6]:
import pandas as pd

def debug_model_outputs(tagger, raw_dataset, num_samples=5):
    """
    Grabs samples from the validation set and compares ground truth to model output.
    """
    print(f"\n--- Model Output Debugging ({num_samples} Samples) ---")
    print(f'[GT = Ground Truth, PR = Model Prediction]')
    
    # Access the original validation data before it was tokenized/stripped
    # We need the 'document' for input and 'tags' for comparison
    val_data = raw_dataset['val']
    
    results = []
    
    for i in range(min(num_samples, len(val_data))):
        sample = val_data[i]
        card_text = sample['document']
        ground_truth = sample['tags']
        
        # Generate prediction using the model's current state
        # generate_tags internally uses self.model.generate()
        predicted_tags = tagger.generate_tags(card_text)
        
        # Format for display
        results.append({
            "Card Text": card_text[:100] + "...", 
            "Ground Truth": ground_truth,
            "Model Prediction": ", ".join(list(predicted_tags)) if predicted_tags else "[EMPTY]"
        })
    
    # Display results
    df = pd.DataFrame(results)
    for idx, row in df.iterrows():
        print(f"\nSample {idx+1}:")
        print(f"  GT: {row['Ground Truth']}")
        print(f"  PR: {row['Model Prediction']}")

# Usage:
# Assuming 'tagger' is your FineTuneLLM instance
debug_model_outputs(tagger, sf.dataset, num_samples=10)


--- Model Output Debugging (10 Samples) ---
[GT = Ground Truth, PR = Model Prediction]

Sample 1:
  GT: feast on the fallen, intervening if clause, repeatable creature tokens, self life loss matters
  PR: activated ability, triggered ability, death trigger-self

Sample 2:
  GT: cost reducer, creaturefall, eminence, one-sided fight, repeatable crime, repeatable removal, spot removal, triggered ability, typal-kavu
  PR: activated ability, triggered ability, cast trigger-self

Sample 3:
  GT: castable from exile, cheaper than mv, delayed trigger, enters in company, exile-self, virtual vanilla
  PR: triggered ability, cast trigger-self

Sample 4:
  GT: auto buyback, cantrip, clash-like, cycle-mor-clashback-spell, draw engine, hand-neutral
  PR: activated ability, triggered ability, player gains mana value

Sample 5:
  GT: attack trigger, cycle-fic-alt-commander, extra combat phase, extra untap, power matters, saboteur, synergy-attacker, synergy-attacker-self, untapper-creature
  PR: attac

In [7]:
record = sf.dataset['test'][1]
pred_tags = tagger.generate_tags(card_text = record['document'])
print(f'Card\n{record["document"]}')
print(f'Actual Tags = {record["tags"]}')
print(f'Predicted Tags = {pred_tags}')

Card

        Generate comma-separated Scryfall community tags for the following card:

        ----------
        Aven Squire
        Mana Cost = {1}{W}
Mana Value = 2.0

        Type Line = Creature — Bird Soldier

        Rules Text = Flying
Exalted (Whenever a creature you control attacks alone, that creature gets +1/+1 until end of turn.)
 
        Power = 1
Toughness = 1

        
        Color Identity = ['W']

        Rarity = common
        ----------
        
Actual Tags = attack trigger, evasion, exalted, french vanilla, royal falcon, synergy-attacker
Predicted Tags = {'attack trigger', 'hate-attacker-self', 'hate-attacker'}
